<a href="https://colab.research.google.com/github/Mengo2414/GomezMatiasPython/blob/main/G%C3%B3mez_Castellano_Mat%C3%ADas_Ejercicio_3_auditoria_web_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 3 - Auditoría y métricas

Primero calculamos emisiones y comprobamos el cálculo con pruebas. Después conectamos esas métricas a una vista Django que presenta una página web.

**Temas:** funciones, parámetros predeterminados, `ValueError`, bucles, diccionarios, `unittest`, JSON y Django.

## 1. Función de emisiones

La fórmula de la consigna es consumo en kWh multiplicado por el factor de emisión. El parámetro vale `0.4` si no se indica otro. Un consumo negativo no tiene sentido para este caso, así que se genera `ValueError`.

In [ ]:
def calcular_emisiones_carbono(consumo_kwh, factor_emision=0.4):
    consumo = float(consumo_kwh)
    factor = float(factor_emision)
    if consumo < 0:
        raise ValueError("El consumo eléctrico no puede ser negativo.")
    return consumo * factor

print(calcular_emisiones_carbono(100))  # 100 * 0.4 = 40.0
print(calcular_emisiones_carbono(50, 0.2))  # factor elegido: 50 * 0.2 = 10.0

40.0
10.0


## 2. Métricas por planta

Cada registro tiene un nombre de planta y su consumo. Recorremos todos los registros para sumar emisiones y agrupar consumos por planta.

In [ ]:
def obtener_metricas_consumo(registros):
    consumo_por_planta = {}
    detalle = []
    emisiones_totales = 0.0

    for registro in registros:
        planta = registro["planta"]
        consumo = float(registro["consumo_kwh"])
        emisiones = calcular_emisiones_carbono(consumo)

        if planta not in consumo_por_planta:
            consumo_por_planta[planta] = 0.0
        consumo_por_planta[planta] = consumo_por_planta[planta] + consumo
        emisiones_totales = emisiones_totales + emisiones
        detalle.append({
            "planta": planta,
            "consumo_kwh": consumo,
            "emisiones_kg_co2": emisiones
        })

    planta_mayor = "Sin datos"
    consumo_mayor = 0.0
    for planta in consumo_por_planta:
        if consumo_por_planta[planta] > consumo_mayor:
            planta_mayor = planta
            consumo_mayor = consumo_por_planta[planta]

    return {
        "emisiones_totales_kg_co2": round(emisiones_totales, 2),
        "planta_mayor_consumo": planta_mayor,
        "consumo_maximo_kwh": round(consumo_mayor, 2),
        "consumo_por_planta": consumo_por_planta,
        "detalle": detalle
    }

In [ ]:
registros = [
    {"planta": "Norte", "consumo_kwh": 12500},
    {"planta": "Sur", "consumo_kwh": 9800},
    {"planta": "Norte", "consumo_kwh": 13100},
    {"planta": "Centro", "consumo_kwh": 15400}
]

metricas = obtener_metricas_consumo(registros)
print(metricas["emisiones_totales_kg_co2"])
print(metricas["planta_mayor_consumo"], metricas["consumo_maximo_kwh"])

20320.0
Norte 25600.0


## 3. Pruebas unitarias con `unittest`

Una prueba compara el resultado esperado con el real. Otra comprueba que el dato inválido produce exactamente la excepción solicitada.

In [ ]:
import unittest

class TestCalculoEmisiones(unittest.TestCase):
    def test_calculo_con_factor_predeterminado(self):
        self.assertAlmostEqual(calcular_emisiones_carbono(100), 40.0)

    def test_calculo_con_factor_personalizado(self):
        self.assertAlmostEqual(calcular_emisiones_carbono(50, 0.2), 10.0)

    def test_consumo_negativo_eleva_value_error(self):
        with self.assertRaises(ValueError):
            calcular_emisiones_carbono(-1)

pruebas = unittest.TestLoader().loadTestsFromTestCase(TestCalculoEmisiones)
unittest.TextTestRunner(verbosity=2).run(pruebas)

test_calculo_con_factor_personalizado (__main__.TestCalculoEmisiones.test_calculo_con_factor_personalizado) ... ok
test_calculo_con_factor_predeterminado (__main__.TestCalculoEmisiones.test_calculo_con_factor_predeterminado) ... ok
test_consumo_negativo_eleva_value_error (__main__.TestCalculoEmisiones.test_consumo_negativo_eleva_value_error) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.007s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>